In [1]:
import torch

In [ ]:
class MyModelLN(torch.nn.Module):
    ######Add this to the previous setup
    #block the entire layer grouping and input that
    class Block(torch.nn.Module):
        def __init__(self, in_channels, out_channels):
            super().__init__()
            self.model = torch.nn.Sequential(
                torch.nn.Linear(in_channels, out_channels),
                torch.nn.LayerNorm(out_channels),
                torch.nn.ReLU(),
                torch.nn.Linear(out_channels, out_channels),
                torch.nn.LayerNorm(out_channels),
                torch.nn.ReLU(),
            )
            if in_channels != out_channels:
                self.skip = torch.nn.Linear(in_channels, out_channels)
            else:
                self.skip = torch.nn.Identity()

        def forward(self, x):
            ##############this is what is added to be a residual self.model(x) is adding the input back [x] 
            ##############but x can't be added because size issue so add it to the model so we add self.model
            return self.skip(x) + self.model(x) 
        
    def __init__(self, layer_size=[512, 512, 512]):
        super().__init__()
        layers = []
        layers.append(torch.nn.Flatten())
        c = 128 * 128 * 3
        layers.append(torch.nn.Linear(c, 512, bias=False))
        c = 512
        for s in layer_size:
            layers.append(self.Block(c, s)) #input the block
            c = s
        layers.append(torch.nn.Linear(c, 102, bias=False))
        self.model = torch.nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


x = torch.randn(10, 3, 128, 128)
net = MyModelLN([512] * 4)
print(net(x))

tensor([[ 0.3513,  1.0360, -0.2476,  ..., -0.0598, -1.3738,  1.5121],
        [ 0.3977,  1.6112, -0.4804,  ..., -0.7547, -2.1287,  1.3343],
        [-0.0498,  0.8150, -0.3630,  ..., -0.6708, -0.4742,  1.4833],
        ...,
        [-0.3625,  1.6688, -1.7643,  ..., -1.5608, -2.2416,  2.5108],
        [ 0.2823,  0.5597, -0.0205,  ..., -0.0590, -1.6264,  0.4328],
        [-1.0035,  1.4273,  1.2379,  ..., -0.6181, -1.5703,  0.7213]],
       grad_fn=<MmBackward0>)


In [ ]:
128 * 128 * 3